# 02. 사물 찾기 직접 추론 — 라이브러리 없이 전처리 · 출력 해석 · 겹친 상자 정리

> edge-lab 서버는 `easydetect` 라이브러리가 이 과정을 대신한다. 여기서는 **OpenVINO 로 모델 계산만 시키고, 나머지는 전부 직접** 짠다.
> 물체 찾기(object detection) 모델이 실제로 무엇을 내놓고, 그것을 어떻게 "상자 몇 개" 로 줄이는지 손으로 따라간다.

쓰는 모델은 **ed-m** — edge-lab 의 [사물 찾기] 가 쓰는 바로 그 모델이다(easydetect, Apache-2.0, COCO 80종).
파이보 · 파이브레인에 넣는 것과 같은 `.onnx` 파일을 OpenVINO 가 바로 읽는다.

### 이 시간에 배우는 것
1. 입력 만들기 — 640 x 640 으로 늘리기 · RGB · 0~1 · NCHW
2. 출력 `boxes [1, 300, 4]` · `scores [1, 300, 80]` 의 뜻 — 300 은 무엇이고 왜 이렇게 적은가
3. 점수 거르기 → 모서리 좌표 → 원래 사진 크기로
4. **IoU** 와 겹친 상자 지우기(**NMS**) — 직접 구현하고, ed 에서는 왜 거의 할 일이 없는지
5. 단계별 시간 재기
6. (덤) 관절 17개 — 사람 상자를 잘라 두 번째 모델에 넣는 **2단계** 방식

### 전체 흐름
```
사진 ─640x640 으로 늘리기─▶ ─RGB·÷255·NCHW─▶ [1,3,640,640]
     ─추론─▶ boxes [1,300,4] (가운데 x, y, 너비, 높이 · 0~1) + scores [1,300,80] (0~1)
     ─점수 거르기─▶ 몇 개 ─모서리 좌표 × 원래 크기─▶ ─겹친 것 정리(IoU 0.7)─▶ 원래 사진 위의 상자
```

## 1. 준비 — 모델과 클래스 이름

In [ ]:
import os
import json
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "..")).resolve()          # 프로그램 폴더 (lab.bat 이 알려 준다)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # 모델 폴더
core = ov.Core()
print("모델 폴더:", MODELS)
print("장치:", core.available_devices)

def pick(*prefer):
    """먼저 적은 장치부터, 이 PC 에 있는 것을 고른다."""
    for d in prefer:
        if d in core.available_devices:
            return d
    return "CPU"

In [ ]:
ED = MODELS / "easydetect" / "ed-m" / "ed-m.onnx"        # setup 이 허깅페이스(leeyunjai/easydetect)에서 받아 둔 것
model = core.read_model(ED)
for i in model.inputs:
    print("입력:", i.any_name, i.partial_shape)
for o in model.outputs:
    print("출력:", o.any_name, o.partial_shape)

# 클래스 이름 — 모델 옆의 ed-m.names.json (COCO 80종)
NAMES = {int(k): v for k, v in json.loads((ED.parent / "ed-m.names.json").read_text(encoding="utf-8")).items()}
print(len(NAMES), "종:", list(NAMES.values())[:8], "...")

**ed-m** — DETR(DEtection TRansformer) 계열의 실시간 물체 찾기 모델. `m` 은 중간 크기(약 2천만 개 숫자),
COCO 라는 공개 사진 묶음으로 학습했다. 입력은 **640 x 640 고정**, 출력은 **두 개**(`boxes`, `scores`)다.

## 2. 사진

In [ ]:
PHOTO = "sample.jpg"          # 웹캠이 없을 때 쓸 사진 (이 노트북과 같은 폴더)

def show(bgr, width=720):
    """OpenCV 사진(BGR)을 노트북 안에 보여 준다."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """웹캠에서 한 장. 웹캠이 없으면 PHOTO 파일."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # 처음 몇 장은 어둡게 나오는 웹캠이 있다
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"웹캠도 {PHOTO} 도 없어요 — PHOTO 에 사진 파일 이름을 넣으세요"
        print("웹캠이 없어 파일을 씁니다:", PHOTO)
    return frame

frame = grab()
print("사진 크기(세로, 가로, 채널):", frame.shape)
show(frame)

## 3. 입력 만들기 — 640 x 640 · RGB · 0~1 · NCHW
입력이 정사각형인데 웹캠 사진은 4:3 이다. 방법은 두 가지가 있다.

| 방법 | 하는 일 | 쓰는 모델 |
|---|---|---|
| **그냥 늘리기** | `cv2.resize(사진, (640, 640))` — 가로 · 세로를 따로 늘린다 | **ed** 가 이렇게 배웠다 |
| letterbox | 비율을 지켜 줄이고 남는 곳을 회색으로 채운다 | YOLO 계열 |

**모델이 배운 방식 그대로 넣어야 한다.** ed 는 늘린 사진으로 배웠으므로 늘려 넣는다. 사람이 조금 홀쭉해 보여도 모델은 그런 사진에 익숙하다.
나중에 상자를 되돌릴 때는 **가로 배율(`w`)과 세로 배율(`h`)이 따로** 다 — 그런데 ed 는 상자를 0~1 비율로 내서, 원래 가로 · 세로를 곱하기만 하면 된다.

ed 는 **RGB** 사진을 **0~1** 로 나눠서 배웠다 (01 번 얼굴 모델은 BGR · 0~255 였다 — 모델마다 다르다!).
`np.ascontiguousarray` 는 축을 바꾼 배열을 메모리에 순서대로 다시 놓는다 (추론 엔진이 더 빨리 읽는다).

In [ ]:
SIZE = 640

def to_blob(bgr, size=SIZE):
    x = cv2.resize(bgr, (size, size))                 # 그냥 늘리기 (ed 가 배운 방식)
    x = cv2.cvtColor(x, cv2.COLOR_BGR2RGB)            # RGB 로 배웠다
    x = x.astype(np.float32) / 255.0                  # 0~255 → 0~1
    x = x.transpose(2, 0, 1)[None]                    # HWC → NCHW
    return np.ascontiguousarray(x)

blob = to_blob(frame)
print(blob.shape, blob.dtype, "값 범위:", blob.min(), "~", blob.max())
show(cv2.resize(frame, (SIZE, SIZE)), 400)            # 모델이 보는 사진 — 조금 늘어나 있다

## 4. 추론과 출력 읽기 — `boxes [1, 300, 4]` · `scores [1, 300, 80]`

**300 은?** ed 는 사진을 보고 **"물체 후보 300 칸"** 을 채운다(이것을 쿼리 query 라고 한다). 칸마다
- `boxes` : `[가운데 x, 가운데 y, 너비, 높이]` — **사진 크기에 대한 비율(0~1)**. 픽셀이 아니다.
- `scores` : 80종 각각의 점수(0~1). 가장 큰 것이 그 칸의 종류 · 확신도다.

물체가 3개뿐인 사진이면 300 칸 중 3 칸만 점수가 높고 나머지는 "아무것도 아님" 으로 낮다.
**ed 는 물체 하나에 칸 하나만 쓰도록 학습했다** — 그래서 같은 물체에 상자가 여러 개 겹쳐 나오는 일이 드물다(6절).

In [ ]:
DEVICE = pick("GPU", "NPU", "CPU")
compiled = core.compile_model(model, DEVICE)
res = compiled(blob)
boxes_n = res[compiled.output("boxes")][0]            # [300, 4] — 가운데 x, y, 너비, 높이 (0~1)
scores_all = res[compiled.output("scores")][0]        # [300, 80]
print(DEVICE, "boxes", boxes_n.shape, "scores", scores_all.shape, "점수 범위", round(float(scores_all.min()), 3), "~", round(float(scores_all.max()), 3))

cls = scores_all.argmax(1)                            # 칸마다 가장 높은 종류
conf = scores_all.max(1)                              # 그 점수
print("점수 0.5 넘는 칸:", int((conf > 0.5).sum()), "/ 300")

top = np.argsort(-conf)[:5]                           # 점수 높은 다섯
for i in top:
    cx, cy, w, h = boxes_n[i]
    print(f"  칸 {i:3d}: {NAMES[int(cls[i])]:12s} 점수 {conf[i]:.2f}  가운데({cx:.2f}, {cy:.2f}) 크기 {w:.2f}x{h:.2f}")

## 5. 점수 거르기 → 모서리 좌표 → 원래 사진 크기로
- **점수 거르기**: ed 는 확신이 없는 칸에도 0.3~0.5 를 준다(YOLO 는 0.25 아래). 그래서 기준을 **0.5** 로 둔다.
- **모서리 좌표**: 그리기 · IoU 는 모서리 `(x1, y1, x2, y2)` 로 한다. `x1 = cx − w/2`, `x2 = cx + w/2` …
- **원래 크기로**: 0~1 비율이므로 **x 에는 원래 가로, y 에는 원래 세로를 곱한다**. 늘려 넣었으니 가로 · 세로 배율이 다르지만 비율이라 곱하기 한 번이면 끝.

$$x_{\text{원래}} = x_{0\sim1} \times \text{가로}, \qquad y_{\text{원래}} = y_{0\sim1} \times \text{세로}$$

In [ ]:
def to_xyxy(b, shape):
    h, w = shape[:2]
    xyxy = np.stack([b[:, 0] - b[:, 2] / 2, b[:, 1] - b[:, 3] / 2,     # 가운데 · 크기 → 모서리
                     b[:, 0] + b[:, 2] / 2, b[:, 1] + b[:, 3] / 2], 1)
    xyxy *= np.array([w, h, w, h], np.float32)                         # 0~1 → 원래 사진 픽셀
    xyxy[:, [0, 2]] = xyxy[:, [0, 2]].clip(0, w)
    xyxy[:, [1, 3]] = xyxy[:, [1, 3]].clip(0, h)
    return xyxy

m = conf > 0.5
xyxy0, cls0, conf0 = to_xyxy(boxes_n[m], frame.shape), cls[m], conf[m]
order = conf0.argsort()[::-1]                                          # 점수 높은 순
xyxy0, cls0, conf0 = xyxy0[order], cls0[order], conf0[order]
for (x1, y1, x2, y2), c, s in zip(xyxy0.astype(int), cls0, conf0):
    print(f"{NAMES[int(c)]:14s} {s:.2f}  ({x1}, {y1}) - ({x2}, {y2})")

## 6. IoU — 두 상자가 얼마나 겹치나
YOLO 같은 모델은 같은 사람을 여러 칸이 동시에 찾아서, 거의 같은 상자가 수십 개 나온다. 이것을 하나로 줄이려면 "얼마나 겹치나" 를 숫자로 재야 한다.

$$\text{IoU} = \frac{\text{겹친 넓이}}{\text{두 상자를 합친 넓이}} = \frac{A \cap B}{A + B - A \cap B}$$

0 이면 전혀 안 겹침, 1 이면 똑같은 상자. 손으로 한 번 계산해 보자.

In [ ]:
def iou(a, b):
    """a: [4] 상자 하나, b: [N, 4] 상자들 (x1, y1, x2, y2)"""
    x1 = np.maximum(a[0], b[:, 0]); y1 = np.maximum(a[1], b[:, 1])     # 겹친 부분의 왼쪽 위
    x2 = np.minimum(a[2], b[:, 2]); y2 = np.minimum(a[3], b[:, 3])     # 겹친 부분의 오른쪽 아래
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)      # 안 겹치면 음수 → 0
    area = lambda r: (r[..., 2] - r[..., 0]) * (r[..., 3] - r[..., 1])
    return inter / (area(a) + area(b) - inter + 1e-9)

A = np.array([0, 0, 100, 100.])
B = np.array([[50, 0, 150, 100.],     # 반쯤 겹침: 겹친 넓이 50x100=5000, 합친 넓이 10000+10000-5000=15000 → 0.333
              [0, 0, 100, 100.],      # 똑같음 → 1
              [200, 200, 300, 300.]]) # 안 겹침 → 0
print(np.round(iou(A, B), 3))

## 7. NMS — 겹친 상자 지우기 (Non-Maximum Suppression)
**알고리즘**
1. 점수 높은 순으로 줄 세운다.
2. 맨 앞(가장 높은 것)을 **남긴다**.
3. 그것과 IoU 가 기준보다 큰 나머지는 "같은 물체" 로 보고 **버린다**.
4. 남은 것들로 2번부터 되풀이.

먼저 만든 상자로 눈으로 확인해 보자.

In [ ]:
def nms(xyxy, scores, iou_th=0.45):
    order = scores.argsort()[::-1]                    # 점수 높은 순
    keep = []
    while len(order):
        i = order[0]
        keep.append(i)                                # 가장 높은 것을 남기고
        rest = order[1:]
        order = rest[iou(xyxy[i], xyxy[rest]) < iou_th]  # 그것과 많이 겹친 것은 버린다
    return np.array(keep, dtype=int)

toy = np.array([[60, 60, 260, 300], [70, 50, 270, 290], [80, 70, 250, 310],   # 같은 물체를 세 번
                [330, 100, 520, 260], [340, 110, 530, 270]], float)           # 다른 물체를 두 번
toy_s = np.array([0.9, 0.8, 0.7, 0.85, 0.6])
kept = nms(toy, toy_s)
canvas = np.full((360, 600, 3), 255, np.uint8)
before, after = canvas.copy(), canvas.copy()
for b, s in zip(toy.astype(int), toy_s):
    cv2.rectangle(before, tuple(b[:2]), tuple(b[2:]), (0, 0, 255), 2)
for k in kept:
    b = toy[k].astype(int)
    cv2.rectangle(after, tuple(b[:2]), tuple(b[2:]), (0, 160, 0), 3)
    cv2.putText(after, f"{toy_s[k]:.2f}", (b[0], b[1] - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 160, 0), 2)
print("남긴 상자:", kept, "(왼쪽: NMS 전 5개, 오른쪽: NMS 뒤)")
show(np.hstack([before, after]), 900)

### 진짜 출력에 적용 — ed 에서는 거의 할 일이 없다
ed 는 물체 하나에 칸 하나만 쓰도록 배웠으므로, 점수 0.5 를 넘는 상자끼리는 대부분 겹치지 않는다.
그래도 가끔 **같은 물체를 두 종류로** 찾는다 — 트럭 0.83 과 승용차 0.57 이 거의 같은 자리(IoU 0.9 이상)에.
그래서 edge-lab 은 **종류와 상관없이 IoU 0.7 넘게 겹치면 점수 낮은 쪽을 지운다** (easydetect 와 같은 규칙).
YOLO 처럼 수십 개를 지우는 게 아니라, 아주 많이 겹친 것 한두 개만 정리하는 셈이다.

In [ ]:
def decode(boxes_n, scores_all, shape, conf_th=0.5, iou_th=0.7):
    cls, conf = scores_all.argmax(1), scores_all.max(1)
    m = conf > conf_th                                                 # ① 점수 거르기
    xyxy, cls, conf = to_xyxy(boxes_n[m], shape), cls[m], conf[m]      # ② 모서리 좌표 · 원래 크기
    keep = nms(xyxy, conf, iou_th)                                     # ③ 많이 겹친 것만 정리 (종류 상관없이)
    return xyxy[keep], cls[keep], conf[keep], int(m.sum())

xyxy, cls_k, conf_k, n_pass = decode(boxes_n, scores_all, frame.shape)
print(f"후보 300 → 점수 통과 {n_pass} → 겹친 것 정리 뒤 {len(xyxy)}")
low = decode(boxes_n, scores_all, frame.shape, conf_th=0.25)
print(f"(기준을 0.25 로 낮추면: 통과 {low[3]} → 정리 뒤 {len(low[0])} — 낮은 점수일수록 같은 물체를 두 번 찾는 일이 생긴다)")

view = frame.copy()
for (x1, y1, x2, y2), c, s in zip(xyxy.astype(int), cls_k, conf_k):
    color = tuple(int(v) for v in np.random.default_rng(int(c)).integers(60, 255, 3))   # 종류마다 같은 색
    cv2.rectangle(view, (x1, y1), (x2, y2), color, 2)
    cv2.putText(view, f"{NAMES[int(c)]} {s:.2f}", (x1, max(18, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
show(view)

## 8. 어디에 시간이 드나
추론만 빠르다고 끝이 아니다. 전처리(크기 바꾸기 · 색 바꾸기 · 나누기)와 후처리(거르기 · 겹친 것 정리)는 장치와 상관없이 **CPU** 가 한다.
GPU · NPU 로 추론이 빨라질수록 전·후처리가 차지하는 몫이 커진다.

In [ ]:
def timed(n=20):
    t = {"전처리": 0.0, "추론": 0.0, "후처리": 0.0}
    for _ in range(n):
        t0 = time.perf_counter(); x = to_blob(frame)
        t1 = time.perf_counter(); r = compiled(x)
        t2 = time.perf_counter(); decode(r[compiled.output("boxes")][0], r[compiled.output("scores")][0], frame.shape)
        t3 = time.perf_counter()
        t["전처리"] += t1 - t0; t["추론"] += t2 - t1; t["후처리"] += t3 - t2
    return {k: v / n * 1000 for k, v in t.items()}

for k, v in timed().items():
    print(f"{k:6s} {v:7.2f} ms  {'█' * max(1, round(v))}")

## 9. (덤) 관절 17개 — 두 번째 모델에 사람만 잘라 넣기 (2단계)
관절 모델 `pose-m` 은 사진 전체가 아니라 **사람 한 명을 잘라 낸 사진(세로 256 x 가로 192)** 을 받는다. 03 번 얼굴 분석과 같은 "찾고 → 잘라서 → 또 묻기" 방식이다.

**자르기** (모델이 배운 방식 그대로)
1. 사람 상자를 세로:가로 = 256:192(4:3) 모양으로 넓히고, 팔다리가 삐져나오니 **1.25 배** 여유를 준다.
2. 그 영역을 256 x 192 로 옮겨 붙인다 — 확대 · 이동을 한 번에 하는 `cv2.warpAffine` (2x3 행렬).
3. 입력은 **RGB 0~255 그대로**(÷255 안 함!) · NCHW. 이 모델은 그렇게 배웠다.

**출력** `x [N, 17, 384]` · `y [N, 17, 512]` — 관절마다 **가로 칸 384개 · 세로 칸 512개의 점수**.
가로 192 픽셀을 반 픽셀씩(×2) 나눈 칸이다. **가장 높은 칸 ÷ 2** 가 잘라 낸 사진 위의 좌표, 그것을 2단계 행렬의 **역변환**으로 원래 사진에 되돌린다.
확신도는 "가장 높은 칸 점수 ÷ 완벽하게 자신 있을 때의 점수" 로 0~1 을 만든다.

In [ ]:
POSE = MODELS / "easydetect" / "pose" / "pose-m.onnx"
SKELETON = [(5, 7), (7, 9), (6, 8), (8, 10), (5, 6), (5, 11), (6, 12), (11, 12),
            (11, 13), (13, 15), (12, 14), (14, 16), (0, 1), (0, 2), (1, 3), (2, 4)]   # COCO 관절 잇기
PH, PW, SPLIT, PAD = 256, 192, 2, 1.25

def crop_matrix(box):
    """사람 상자 → 256x192 로 옮기는 2x3 행렬 (4:3 으로 넓히고 1.25 배 여유)."""
    x1, y1, x2, y2 = box
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    w, h = max(x2 - x1, 1.0), max(y2 - y1, 1.0)
    if w > h * PW / PH: h = w * PH / PW          # 4:3 모양으로 — 모자란 쪽을 늘린다
    else:               w = h * PW / PH
    w, h = w * PAD, h * PAD
    sx, sy = PW / w, PH / h
    return np.array([[sx, 0, PW / 2 - sx * cx], [0, sy, PH / 2 - sy * cy]])

def peak(sigma, bins):
    """완벽하게 자신 있을 때(학습 목표 모양) 가장 높은 칸의 점수 — 확신도의 기준"""
    d = np.arange(bins) - bins / 2
    g = np.exp(-d ** 2 / (2 * sigma ** 2))
    return g.max() / g.sum()

def softmax(z):
    z = z - z.max(-1, keepdims=True); e = np.exp(z)
    return e / e.sum(-1, keepdims=True)

if not POSE.exists():
    print("관절 모델이 없어 건너뜁니다:", POSE)
else:
    pose = core.compile_model(POSE, DEVICE)
    people = xyxy[cls_k == 0]                                               # 7절에서 찾은 사람만
    mats = [crop_matrix(b) for b in people]
    crops = [cv2.cvtColor(cv2.warpAffine(frame, mt, (PW, PH)), cv2.COLOR_BGR2RGB) for mt in mats]
    view = frame.copy()
    if crops:
        x_in = np.ascontiguousarray(np.stack(crops).transpose(0, 3, 1, 2).astype(np.float32))   # RGB 0~255 · NCHW
        r = pose(x_in)
        xl, yl = r[pose.output("x")], r[pose.output("y")]                    # [N,17,384] · [N,17,512]
        px, py = softmax(xl), softmax(yl)
        u, v = px.argmax(-1) / SPLIT, py.argmax(-1) / SPLIT                  # 잘라 낸 사진 위의 좌표
        kc = np.minimum(px.max(-1) / peak(4.9, PW * SPLIT), py.max(-1) / peak(5.66, PH * SPLIT)).clip(0, 1)
        for mt, uu, vv, cc in zip(mats, u, v, kc):
            sx, tx, sy, ty = mt[0, 0], mt[0, 2], mt[1, 1], mt[1, 2]
            pts = np.stack([(uu - tx) / sx, (vv - ty) / sy], 1)              # 역변환 — 원래 사진으로
            for a, c in SKELETON:
                if cc[a] > 0.3 and cc[c] > 0.3:                              # 둘 다 보일 때만 잇는다
                    cv2.line(view, tuple(pts[a].astype(int)), tuple(pts[c].astype(int)), (255, 160, 0), 3)
            for (x, y), k in zip(pts, cc):
                if k > 0.3:
                    cv2.circle(view, (int(x), int(y)), 4, (0, 0, 255), -1)
        show(np.hstack([cv2.cvtColor(c, cv2.COLOR_RGB2BGR) for c in crops[:4]]), 600)   # 모델이 본 사람 사진
    print("사람", len(crops), "명")
    show(view)

## 자주 나는 오류
| 증상 | 원인 · 고침 |
|---|---|
| 아무것도 못 찾음 | `÷255` 를 빠뜨렸거나 BGR 그대로 넣었다 — 3절 확인 |
| 상자가 엉뚱한 곳 · 아주 작음 | 0~1 비율을 원래 가로 · 세로로 안 곱했다 — 5절 |
| 점수가 이상하게 낮음(0.7 대) | 이미 0~1 인 점수에 시그모이드를 또 씌웠다 — 그대로 쓴다 |
| 같은 자리에 트럭 · 승용차가 겹침 | 겹친 것 정리(IoU 0.7)를 빠뜨렸다 — 7절 |
| 관절이 전부 사진 왼쪽 위에 몰림 | 역변환을 빠뜨렸다(잘라 낸 사진 좌표 그대로) — 9절 |
| 관절을 하나도 못 찾음 | 관절 모델에 ÷255 를 했다 — 이 모델은 0~255 그대로 |

## 정리
- 전처리: 640 x 640 으로 늘리기(모델이 배운 방식) → RGB → ÷255 → NCHW.
- 출력: 후보 300 칸 — 칸마다 상자(0~1 비율) + 80종 점수(이미 0~1). 물체 하나에 칸 하나.
- 후처리: 점수 거르기(0.5) → 모서리 좌표 × 원래 크기 → 많이 겹친 것만 정리(IoU 0.7).
- 관절: 사람 상자 → 4:3 · 1.25 배로 잘라 → 칸 점수 최댓값 ÷ 2 → 역변환.
- 시간: 추론이 빨라질수록 전·후처리 몫이 커진다.

## 해 볼 것
1. `decode(..., conf_th=0.3)` 로 낮추고, `iou_th=1.0`(정리 안 함)으로도 바꿔 보자. 상자 수가 어떻게 달라지나?
2. letterbox(비율 지키고 회색 채우기)로 넣으면 결과가 어떻게 달라지나? 모델이 배운 방식과 다르게 넣으면 점수가 떨어지는지 확인해 보자.
3. `DEVICE` 를 `"CPU"` · `"NPU"` 로 바꿔 8절 표를 비교해 보자.
4. 라이브러리 결과와 맞는지 확인해 보자.
   <details><summary>힌트</summary>

   `from easydetect import Detector; r = Detector(str(ED), device="CPU")(frame, conf=0.5)[0]; print(r.boxes.xyxy, r.boxes.cls, r.boxes.conf)`
   </details>
5. 웹캠 영상에서 "사람" 만 세어 화면에 숫자로 띄워 보자 (`cls_k == 0`).

---
# 🛠 미니 프로젝트 — 사람 수 세기 모니터

**만들 것**: 웹캠 영상에서 **사람(종류 0)만** 세어 화면에 띄우고, 1초마다 시각 · 사람 수를 **CSV** 로 기록한다.
끝나면 가장 많았던 순간과 평균을 알려 준다. (매장 · 교실 혼잡도 측정의 아주 작은 판)

**쓰는 것**: `to_blob` · `decode` (이 노트북 — 라이브러리 없이 만든 것 그대로), `live`

In [ ]:
from IPython.display import Image as IPImage

def live(process, seconds=15, index=0, width=640):
    """웹캠 영상을 seconds 초 동안 process(사진) → 사진 으로 바꿔 노트북 안에 보여 준다. 멈추려면 I I."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    still = None if cap.isOpened() and cap.read()[0] else cv2.imread(PHOTO)   # 웹캠이 없으면 사진으로
    handle = display(None, display_id=True)
    t0, n, out = time.time(), 0, None
    try:
        while time.time() - t0 < seconds and (still is None or n < 10):
            if still is None:
                ok, f = cap.read()
                if not ok:
                    break
            else:
                f = still.copy()
            out = process(f)
            s = width / out.shape[1]
            small = cv2.resize(out, (width, int(out.shape[0] * s))) if s < 1 else out
            handle.update(IPImage(data=cv2.imencode(".jpg", small)[1].tobytes()))
            n += 1
    except KeyboardInterrupt:
        pass
    finally:
        cap.release()
    print(f"{n}장 · {n / max(time.time() - t0, 1e-6):.1f} FPS" + (" (웹캠이 없어 사진으로 흉내)" if still is not None else ""))
    return out

In [ ]:
import csv
from datetime import datetime

log, last_log = [], [0.0]

def count_people(f):
    r = compiled(to_blob(f))
    xyxy, c, s, _ = decode(r[compiled.output("boxes")][0], r[compiled.output("scores")][0], f.shape)
    people = xyxy[c == 0]                                           # 사람만
    for x1, y1, x2, y2 in people.astype(int):
        cv2.rectangle(f, (x1, y1), (x2, y2), (0, 200, 0), 2)
    cv2.putText(f, f"people: {len(people)}", (10, 36), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 0, 255), 3)
    if time.time() - last_log[0] >= 1.0:                            # 1초에 한 번 기록
        log.append((datetime.now().strftime("%H:%M:%S"), len(people)))
        last_log[0] = time.time()
    return f

live(count_people, seconds=20)
with open("people_log.csv", "w", newline="", encoding="utf-8") as fp:
    csv.writer(fp).writerows([("time", "people")] + log)
if log:
    counts = [n for _, n in log]
    peak_ = max(log, key=lambda x: x[1])
    print(f"기록 {len(log)}개 → people_log.csv · 평균 {np.mean(counts):.1f}명 · 최대 {peak_[1]}명 ({peak_[0]})")

**더 해 보기**
- 사람 대신 다른 종류 세기 (예: 병 39, 컵 41 — `NAMES` 에서 번호 찾기)
- 화면 왼쪽 절반 / 오른쪽 절반의 사람 수를 따로 세기 (상자 가운데 x 로 나누기)
- 사람이 0 명 → 1 명 이상으로 바뀌는 순간만 기록하기 (출입 알림)